In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [ ]:
# experimental data
save_folder = 'run7'
n_points = 10

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [ ]:
# def model functions 
# --- Model functions, all using q2 notation --- 

def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)


def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))


def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))


# --- Amplitudes T1 and T2 rewritten with strict q2 notation ---

def T_1(k, q2, phi, mg, a1, a2, m2_func):
    q  = np.sqrt((q2))     # this is |q|
    k2 = k ** 2

    qk_cos = q * (k) * np.cos(phi)

    qk_plus_squared  = q2/4 + qk_cos + k2
    qk_minus_squared = q2/4 - qk_cos + k2

    alpha_D_plus  = alpha_D(qk_plus_squared,  mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * (G0 ** 2)


def T_2(k, q2, phi, mg, a1, a2, m2_func):
    q  = np.sqrt((q2))
    k2 = k ** 2

    qk_cos = q * (k) * np.cos(phi)

    qk_plus_squared  = q2/4 + qk_cos + k2
    qk_minus_squared = q2/4 - qk_cos + k2

    alpha_D_plus  = alpha_D(qk_plus_squared,  mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k2 - q2/4)

    G0     = G_p(q2,    a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value * amp_value.conjugate()
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [ ]:
def full_int(
    mg,
    a1,
    a2,
    m2_func,
    q2_val,
    sqrt_s,
    epsabs=1e-10,
    epsrel=1e-10,
    limit=200
):
    q2_val = np.atleast_1d(q2_val)

    out = []

    for q2 in q2_val:

        def integrand(u, y):
            # Remapping
            x = u * u
            k = sqrt_s * x
            phi = 2.0 * np.pi * y

            # Measure
            dx_weight = 2.0 * u**3          # x dx
            scale = 2.0 * np.pi * sqrt_s    # angular Jacobian

            val = (
                T_1(k, q2, phi, mg, a1, a2, m2_func)
                - T_2(k, q2, phi, mg, a1, a2, m2_func)
            )

            # IMPORTANT: restore √s from k
            return dx_weight * sqrt_s * np.real(val) * scale

        def inner(u):
            res, _ = quad(
                lambda y: integrand(u, y),
                0.0, 1.0,
                epsabs=epsabs,
                epsrel=epsrel,
                limit=limit
            )
            return res

        I_re, err_re = quad(
            inner,
            0.0, 1.0,
            epsabs=epsabs,
            epsrel=epsrel,
            limit=limit
        )

        I_im = 0.0
        err_im = 0.0

        out.append((I_re, I_im, err_re, err_im))

    if len(out) == 1:
        return out[0]

    return tuple(np.array(v) for v in zip(*out))

In [ ]:
diff_t_born = []

def get_dif_sigma(
    epsilon,
    mg,
    a1,
    a2,
    mg_model,
    tol_abs=1e-8,
    tol_rel=1e-6,
    quad_limit=200
):
    """
    Differential sigma using real/imag separated integrals.
    """

    sqrt_s = 7000.0
    scale = 1.0

    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.005

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        I_re, I_im, err_re, err_im = full_int(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=mg_model,
            q2_val=q2,
            sqrt_s=sqrt_s,
            epsabs=tol_abs,
            epsrel=tol_rel,
            limit=quad_limit
        )

        print(
            f"q2={q2:.4f} | "
            f"I_re={I_re:.6e} ± {err_re:.2e} | "
            f"I_im={I_im:.6e} ± {err_im:.2e}"
        )

        diff_T = I_re + 1j * I_im
        diff_t_born.append(diff_T)

        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, epsilon, t)
        dif_sigma = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}


In [ ]:

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    ensemble_parameters["atlas"]["pl"]['epsilon'],
    ensemble_parameters["atlas"]["pl"]['mg'], 
    ensemble_parameters["atlas"]["pl"]['a1'],
    ensemble_parameters["atlas"]["pl"]['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [ ]:


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, np.real(dif_sigma_pl_atlas_7_values),label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [ ]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()


def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))


In [ ]:
lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13001
step = 100


def get_sigma_tot(
    epsilon,
    mg,
    a1,
    a2,
    mg_model,
    tol_abs=1e-8,
    tol_rel=1e-6,
    quad_limit=200
):
    """
    Total cross section using stabilized nested quad integration
    at q² = 0.
    """

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:

        s = sqrt_s ** 2

        # ------------------------------------------------------------
        # High-precision stabilized integration at q² = 0
        # ------------------------------------------------------------
        I_re, I_im, err_re, err_im = full_int(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=mg_model,
            q2_val=0.0,
            sqrt_s=sqrt_s,
            epsabs=tol_abs,
            epsrel=tol_rel,
            limit=quad_limit
        )

        print(
            f"sqrt(s)={sqrt_s:6.0f} | "
            f"I_re={I_re:.6e} ± {err_re:.2e}"
        )

        # Physical amplitude (imag part is zero here)
        born_amp = amp_calculation(I_re, s, epsilon, 0.0)
        lst_born_amp.append(born_amp)

        lst_sigma_tot.append(
            sigma_tot(born_amp, s)
        )

        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step

    return lst_sigma_tot, lst_sqrt_s


In [ ]:

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    ensemble_parameters["atlas"]["pl"]['epsilon'],
    ensemble_parameters["atlas"]["pl"]['mg'], 
    ensemble_parameters["atlas"]["pl"]['a1'],
    ensemble_parameters["atlas"]["pl"]['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]


In [ ]:



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")